# Data Quality Scoring

> **Purpose**: Demonstrate the `scoring.py` module — compute a multi-dimensional quality score using real pipeline outputs.

**All business logic lives in `../scoring.py`. This notebook only imports and calls those functions.**

### Quality Dimensions
| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| **Completeness** | 40% | % of non-null values across all cells |
| **Uniqueness** | 30% | % of non-duplicate rows |
| **Validity** | 30% | % of records free from rule violations + anomalies |
| **Rules Quality Score** | — | 0–100 score from `rules.py` (shown separately) |
| **Anomaly Penalty** | — | % of rows flagged as anomalous (informational) |

**Overall = Completeness × 0.40 + Uniqueness × 0.30 + Validity × 0.30**

> **Previous step**: `anomaly.ipynb` | **Next step**: `reports.ipynb`

In [ ]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from cleaning import load_dataset, run_cleaning
from validation import run_validation
from rules import run_business_rules
from anomaly import run_ml_anomalies
from scoring import (
    compute_dataset_scores,
    compute_column_scores,
    compute_record_scores,
    run_scoring,
)

DATA_PATH = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
df_raw   = load_dataset(DATA_PATH)
df_clean = run_cleaning(df_raw)
print(f"Dataset loaded and cleaned: {df_clean.shape}")

## Collect Pipeline Outputs

Scoring needs the outputs from validation, rules, and anomaly detection to compute a fully data-driven validity score.

In [ ]:
val_result     = run_validation(df_clean)
violations     = run_business_rules(df_clean)
anomaly_result = run_ml_anomalies(df_clean)

print(f"Validation violations    : {len(val_result['violations'])}")
print(f"Business rule violations : {len(violations)}")
print(f"Consensus anomalies      : {anomaly_result.get('consensus_anomalies', 0):,}")

## Dataset-Level Quality Scores

In [ ]:
dataset_scores = compute_dataset_scores(
    df_clean,
    val_result=val_result,
    violations=violations,
    anomaly_result=anomaly_result,
)

print("=== Dataset Quality Scores ===")
for k, v in dataset_scores.items():
    print(f"  {k:<25}: {v}")

## Column-Level Completeness Scores

In [ ]:
col_scores = compute_column_scores(df_clean)
scores_df = pd.DataFrame.from_dict(col_scores, orient="index", columns=["Completeness %"])
scores_df.sort_values("Completeness %", ascending=True)

## Per-Record Score Distribution

In [ ]:
record_scores = compute_record_scores(df_clean)
print("Per-record completeness score distribution:")
print(record_scores.describe().round(2))
print(f"\nRows with score < 50%  : {(record_scores < 50).sum():,}")
print(f"Rows with score = 100% : {(record_scores == 100).sum():,}")

## Full Scoring Run

In [ ]:
scores_report = run_scoring(
    df_clean,
    val_result=val_result,
    violations=violations,
    anomaly_result=anomaly_result,
)

print("=== Final Quality Scores ===")
print(json.dumps(scores_report["dataset"], indent=2))

print("\n=== Record Score Statistics ===")
print(json.dumps(scores_report["record_score_stats"], indent=2))

---
## Key Takeaways

- **`dataset_score`** is the weighted average of Completeness, Uniqueness, and Validity
- **`rules_quality_score`** shows what fraction of records pass all business rules — reported separately from the main score
- **`anomaly_penalty`** (in %) shows the ML-detected anomaly rate — it is informational and does not affect the weighted formula directly
- **`validity_score`** is computed as `(1 − violated_records / total_records) × 100` — lower means more records have issues
- Completeness has the highest weight (40%) because missing data is the most common data quality problem
- Columns sorted by completeness (ascending) identify the most data-sparse fields first